# Fine-tuning Mistral Models for Patent Classification

This notebook is used to fine-tune Mistral models via their fine-tuning API, aiming to improve patent classification performance over prompt-based methods.

## Process Overview
1. **Setup**: Configure training parameters and file paths
2. **Data Upload**: Upload training and validation datasets to Mistral
3. **Job Creation**: Create and validate fine-tuning job
4. **Training**: Monitor training progress with loss curves
5. **Inference**: Test fine-tuned model on test set
6. **Evaluation**: Generate results for performance assessment

In [ ]:
import os
import json
import pandas as pd
import time
from tqdm.notebook import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt
from mistralai import Mistral
import asyncio
from concurrent.futures import ThreadPoolExecutor

# Configuration
ds_id = "ds3"  # dataset id of training data
job_type = "classifier" # classifier or completion

if job_type == "completion":
    is_chat = "_chat"
else:
    is_chat = ""
    
train_file_path = f"data/ft_{ds_id}_train{is_chat}.jsonl"
val_file_path = f"data/ft_val{is_chat}.jsonl"
test_file_path = f"data/ft_test.jsonl"
test_results_path = f"results/ft_test{is_chat}_{ds_id}_results.jsonl"

ft_run_name = f"{ds_id}_8b{is_chat}"
wandb_project_name = "cpc"

lr = 0.00001
n_epochs = 1
model_to_ft = "ministral-3b-latest"

In [ ]:
client = Mistral(api_key=os.environ.get("MISTRAL"))

# Upload training data
training_data = client.files.upload(
    file={
        "file_name": train_file_path.split("/")[-1],
        "content": open(train_file_path, "rb"),
    }
)
# Upload validation data
validation_data = client.files.upload(
    file={
        "file_name": val_file_path.split("/")[-1],
        "content": open(val_file_path, "rb"),
    }
)
print("Uploaded training and validation data")

In [ ]:
# Create a fine-tuning job
created_job = client.fine_tuning.jobs.create(
    model=model_to_ft,
    job_type=job_type,
    training_files=[{"file_id": training_data.id, "weight": 1}],
    validation_files=[validation_data.id],
    suffix=ds_id,
    hyperparameters={"epochs": n_epochs, "learning_rate": lr},
    auto_start=False,
    integrations=[
        {
            "project": wandb_project_name,
            "api_key": os.environ.get("WANDB_API_KEY"),
            "run_name": ft_run_name,
        }
    ],
)
print(f"Created fine-tuning job: {created_job.id}")

In [ ]:
# Retrieve the job details
retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)
print(json.dumps(retrieved_job.model_dump(), indent=4))

# Wait for the job to be validated
while retrieved_job.status not in ["VALIDATED"]:
    retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)

    clear_output(wait=True)  # Clear the previous output
    print("Waiting for job to be validated...")
    print(json.dumps(retrieved_job.model_dump(), indent=4))
    time.sleep(1)
clear_output()

expected_duration_minutes = retrieved_job.metadata.expected_duration_seconds / 60
expected_duration_hours = expected_duration_minutes / 60
print(
    f"Job validated. Expected duration: {expected_duration_hours:.1f} hours ({expected_duration_minutes:.0f} minutes). Cost in {retrieved_job.metadata.cost_currency}: {retrieved_job.metadata.cost}"
)

In [ ]:
# Start the fine-tuning job
client.fine_tuning.jobs.start(job_id=created_job.id)

# Retrieve the job details again
retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)
print(f"Job status: {retrieved_job.status}")
print(json.dumps(retrieved_job.model_dump(), indent=4))

## Training Progress Monitoring

Real-time visualization of training and validation loss curves. (Code obtained from Mistral's Product Classifier Cookbook.)

In [ ]:
# Plot loss curve
# Initialize DataFrames to store the metrics
train_metrics_df = pd.DataFrame(columns=["Step Number", "Train Loss"])
valid_metrics_df = pd.DataFrame(columns=["Step Number", "Valid Loss"])

# Total training steps
total_training_steps = retrieved_job.hyperparameters.training_steps

# Wait for the job to complete
while retrieved_job.status in ["QUEUED", "RUNNING"]:
    retrieved_job = client.fine_tuning.jobs.get(job_id=created_job.id)

    if retrieved_job.status == "QUEUED":
        time.sleep(5)
        continue

    # Clear the previous output
    clear_output(wait=True)
    print(retrieved_job.status)

    # Extract metrics from all checkpoints
    for checkpoint in retrieved_job.checkpoints[::-1]:
        metrics = checkpoint.metrics
        step_number = checkpoint.step_number

        # Check if the step number is already in the DataFrame
        if step_number not in train_metrics_df["Step Number"]:
            # Prepare the new row for train loss
            train_row = {
                "Step Number": step_number,
                "Train Loss": metrics.train_loss,
            }

            # Append the new train metrics to the DataFrame
            train_metrics_df = pd.concat(
                [train_metrics_df, pd.DataFrame([train_row])], ignore_index=True
            )

            # Prepare the new row for valid loss if available
            if metrics.valid_loss != 0:
                valid_row = {
                    "Step Number": step_number,
                    "Valid Loss": metrics.valid_loss,
                }
                # Append the new valid metrics to the DataFrame
                valid_metrics_df = pd.concat(
                    [valid_metrics_df, pd.DataFrame([valid_row])], ignore_index=True
                )

    if len(retrieved_job.checkpoints) > 0:
        # Sort the DataFrames by step number
        train_metrics_df = train_metrics_df.sort_values(by="Step Number")
        valid_metrics_df = valid_metrics_df.sort_values(by="Step Number")

        # Plot the evolution of train loss and valid loss
        plt.figure(figsize=(10, 6))

        # Plot train loss
        plt.plot(
            train_metrics_df["Step Number"],
            train_metrics_df["Train Loss"],
            label="Train Loss",
            linestyle="-",
        )

        # Highlight start and end points of train loss
        plt.scatter(
            train_metrics_df.iloc[[0, -1]]["Step Number"],
            train_metrics_df.iloc[[0, -1]]["Train Loss"],
            color="blue",
            zorder=5,
        )

        # Plot valid loss only if available
        if not valid_metrics_df.empty:
            plt.plot(
                valid_metrics_df["Step Number"],
                valid_metrics_df["Valid Loss"],
                label="Valid Loss",
                linestyle="--",
            )

            # Highlight start and end points of valid loss
            plt.scatter(
                valid_metrics_df.iloc[[0, -1]]["Step Number"],
                valid_metrics_df.iloc[[0, -1]]["Valid Loss"],
                color="orange",
                zorder=5,
            )

        plt.xlabel("Step Number")
        plt.ylabel("Loss")
        plt.title("Train Loss and Valid Loss")
        plt.legend()
        plt.grid(True)
        plt.show()

    time.sleep(1)

## Model Inference and Evaluation

Test the fine-tuned model on the test set using the classifier API. Results are saved in JSONL format for evaluation with `eval.py`.

In [ ]:
# To set
model_id = "" # or use `retrieved_job.fine_tuned_model`
is_chat = True
sys_prompt_file = "prompts/sys_prompt_train.md" # only when is_chat is True
user_prompt_file = "prompts/user_prompt_train.md" # only when is_chat is True

# Load the test samples
with open(test_file_path, "r") as f:
    test_samples = [json.loads(line) for line in f.readlines()]


def classify_sample(sample_text, model_id, is_chat=False, sys_prompt=None, user_prompt=None):

    if not is_chat:
        classifier_response = client.classifiers.classify(
            model=model_id,
            inputs=[sample_text],
        )
        # Extract class IDs from classifier response
        result = classifier_response.model_dump()
        class_ids = []
        
        if 'results' in result and result['results']:
            classification = result['results'][0]
            if 'cpc_class_ids' in classification:
                scores = classification['cpc_class_ids']["scores"]
                class_ids.extend([k for k, v in scores.items() if v > 0.05])
        return class_ids
    else:
        # Extract class IDs from chat response
        response = client.chat.complete(
            model=model_id,
            messages=[
                {"role": "system", "content": sys_prompt},
                {
                    "role": "user",
                    "content": user_prompt.format(
                            patent_description=sample_text
                        ),
                    },
                ],
                temperature=0,
            )
        
        response_content = response.choices[0].message.content

        # Clean up response content (remove markdown formatting)
        response_content = response_content.replace("```json\n", "").replace(
            "\n```", ""
        )

        # Convert response to JSON array
        try:
            return json.loads(response_content)
        except json.JSONDecodeError:
            print(f"Invalid JSON response: {response_content}")
            return []

sys_prompt = open(sys_prompt_file, "r").read() if is_chat else None
user_prompt = open(user_prompt_file, "r").read() if is_chat else None

# Classify the first test sample
classifier_response = classify_sample(test_samples[0]["text"], model_id, is_chat=is_chat, sys_prompt=sys_prompt, user_prompt=user_prompt)

print("Predicted class IDs:", classifier_response)
print("Patent description:", test_samples[0]["text"])

In [ ]:
# Classify all test samples
test_results = []
CONCURRENCY = 5
SLEEP_BETWEEN_REQUESTS = 0.5  # seconds

async def classify_sample_async(text, model_id, is_chat, sys_prompt, user_prompt):
    return await asyncio.to_thread(classify_sample, text, model_id, is_chat, sys_prompt, user_prompt)

async def run_async_classification(test_samples):
    semaphore = asyncio.Semaphore(CONCURRENCY)

    async def sem_task(i, sample):
        async with semaphore:
            predicted_class_ids = await classify_sample_async(sample["text"], model_id, is_chat, sys_prompt, user_prompt)
            test_results.append({
                "custom_id": i,
                "pred_class_ids": json.dumps(predicted_class_ids)
            })
            await asyncio.sleep(SLEEP_BETWEEN_REQUESTS)

    tasks = [sem_task(i, sample) for i, sample in enumerate(test_samples)]
    for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        await f

await run_async_classification(test_samples)

# Store in results jsonl file
with open(test_results_path, "w") as f:
    for result in test_results:
        f.write(json.dumps(result) + "\n")
print(f'To evaluate, run `python eval.py -f "ft" -d {ds_id} -r {test_results_path} -m {model_id}`')